# Xiaofeiyu-12B-v4 · Google Colab 免费 T4 (15 GB) 一键体验

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niatheelf777/xiaofeiyu/blob/main/notebooks/xiaofeiyu_colab_free_t4.ipynb)

在 Google Colab 免费 **NVIDIA T4 (15.0 GiB)** 上零配置运行 **Xiaofeiyu-12B-v4**：
- **显存占用**：`gemma-4-12B-it-UD-Q4_K_XL-xfy` (`7.72 GiB`) + 5 枚 `.gguf` 人格声带卡带 + 8k 上下文，总占用约 **`~8.9 GiB`**（轻松稳跑在 15.0 GiB T4 内）。
- **Split-Phase 脑声解耦**：思考（`<|channel>thought`）、沙箱算账与代码生成全程保持 **`LoRA = 0.0`** 原生满血智商；仅在发送气泡时实时激活五维声带权重。
- **合规内嵌网页**：直接通过 Colab 官方 `serve_kernel_port_as_iframe` 在 Notebook 内渲染二维性格球盘与微信短气泡界面，不使用任何外部穿透隧道工具。

### 五维人格槽位（Five Persona Slots）

| 槽位 | 分支代号 | 中文名称 | English Name | 风格定位 |
| :---: | :--- | :--- | :--- | :--- |
| `s0` | `xiaofeiyu_backbone` | **元气日常** | Everyday Warmth | 自然松弛的日常朋友搭子，短句接话不端架子 |
| `c1` | `neko_clingy` | **甜系软萌** | Sweet & Clingy | 软糯粘人、直球撒娇的高温度陪伴 |
| `c2` | `older_tease` | **知性姐姐** | Poised Older Sister | 从容温柔、游刃有余的姐姐感与包纳感 |
| `c3` | `brat_mesugaki` | **傲娇毒舌** | Tsundere Wit | 嘴硬心软、欢喜冤家式的机锋吐槽 |
| `c4` | `fragile_porcelain` | **安静慢热** | Quiet & Gentle | 轻声细语、细腻克制的慢热倾听 |


In [ ]:
#@title 1. 检查 T4 GPU、拉取代码与权重并后台启动 Split-Phase 网关
import json
from pathlib import Path
import subprocess
import sys
import time
import urllib.error
import urllib.request

# 1) Verify GPU runtime is attached
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError(
        "未检测到 NVIDIA GPU！请在 Colab 菜单栏选择 Runtime -> Change runtime type -> T4 GPU 后重试。"
    )
print("Detected GPU:", gpu_check.stdout.strip())

# 2) Clone or update repository
repo_dir = Path("/content/xiaofeiyu")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/niatheelf777/xiaofeiyu.git", str(repo_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)

# 3) Install minimal Python dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tokenizers", "huggingface_hub"],
    check=True,
)

# 4) Stop any previous instance cleanly before launching
for pattern in ("launch_local.py", "pc2_split_phase_server.py", "llama-server"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(0.5)

log_path = Path("/content/xiaofeiyu_server.log")
log_file = log_path.open("w", encoding="utf-8")
cmd = [
    sys.executable,
    "server/launch_local.py",
    "--preset",
    "12b-q4",
    "--auto-download-llama",
]
server_proc = subprocess.Popen(
    cmd,
    cwd=str(repo_dir),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
print(f"Started background launcher (PID={server_proc.pid}); pulling weights & starting llama-server...")
print(f"Live log: {log_path}")

# 5) Poll http://127.0.0.1:8081/api/status until healthy
status_url = "http://127.0.0.1:8081/api/status"
deadline = time.monotonic() + 600.0
last_report = 0.0
while time.monotonic() < deadline:
    if server_proc.poll() is not None:
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(
            f"launch_local.py 意外退出 (code={server_proc.returncode})。最近日志：\n{tail}"
        )
    try:
        with urllib.request.urlopen(status_url, timeout=2.0) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode("utf-8"))
                if data.get("ready"):
                    print("Ready! Xiaofeiyu-12B-v4 (12b-q4) Split-Phase Gateway is healthy on :8081.")
                    print(json.dumps(data, ensure_ascii=False, indent=2))
                    break
    except (OSError, urllib.error.URLError, json.JSONDecodeError):
        pass
    now = time.monotonic()
    if now - last_report >= 15.0:
        print("Waiting for model download / llama-server warmup...")
        last_report = now
    time.sleep(2.0)
else:
    tail = log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
    raise TimeoutError(f"等待服务就绪超时 (600s)。最近日志：\n{tail}")


In [ ]:
#@title 2. 在 Colab 内直接打开二维性格球盘 WebUI（合规免隧道内嵌）
from IPython.display import HTML, display

try:
    from google.colab import output
except ImportError as exc:
    raise RuntimeError("请在 Google Colab 环境中运行此单元格。") from exc

print("正在内嵌 Xiaofeiyu 二维性格球盘（拖动球盘实时混调：元气日常 / 甜系软萌 / 知性姐姐 / 傲娇毒舌 / 安静慢热）：")
output.serve_kernel_port_as_iframe(8081, path="/", height="780")

try:
    proxy_url = output.eval_js("google.colab.kernel.proxyPort(8081)")
    display(
        HTML(
            f'<div style="margin-top:8px;font-family:sans-serif;font-size:13px;">'
            f'若内嵌框未加载，可点击当前 Colab 会话专属认证链接（仅限本人账号访问）：'
            f'<a href="{proxy_url}" target="_blank" rel="noopener">{proxy_url}</a></div>'
        )
    )
except Exception as exc:
    print(f"生成备用认证链接跳过: {exc}")


In [ ]:
#@title 3. 可选：Python OpenAI 兼容接口与五滑条自定义混音测试
import json
import urllib.request

# 权重顺序: [s0 元气日常, c1 甜系软萌, c2 知性姐姐, c3 傲娇毒舌, c4 安静慢热]
# 示例：1.0 元气日常 + 0.4 甜系软萌 + 0.6 知性姐姐
payload = {
    "model": "xiaofeiyu-v4|1.0,0.4,0.6,0,0",
    "temperature": 0.85,
    "top_p": 0.95,
    "stream": False,
    "messages": [
        {
            "role": "user",
            "content": "刚加完班出地铁，风好大，写个三行 Python 算一下 2 的 10 次方顺便安慰我一句",
        }
    ],
}

req = urllib.request.Request(
    "http://127.0.0.1:8081/v1/chat/completions",
    data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=120.0) as resp:
    result = json.loads(resp.read().decode("utf-8"))

print("Model tag:", result.get("model"))
print("Reply:\n" + result["choices"][0]["message"]["content"])
